# E5 (2022)
---
[[paper]](https://arxiv.org/abs/2212.03533)<br>
**E5** = **E**mb**E**ddings from bidirection **E**ncoder representations (в названии обыгрывается наличие пяти букв «E»).

E5 — это семейство моделей (Small, Base, Large) для генерации универсальных Text Embeddings, разработанное командой Microsoft. Метод примечателен тем, что он первым показал, как использование простых текстовых инструкций-префиксов внутри стандартного Encoder позволяет эффективно решать проблему асимметричного поиска.

**Контекст**
До появления E5 большинство моделей для Dense Retrieval либо обучались на узких supervised-датасетах (как DPR), что ограничивало их Generalization, либо использовали Unsupervised Contrastive Learning (как SimCSE), что хорошо работало для Semantic Textual Similarity (STS), но плохо для реального поиска, где запрос и документ имеют разную структуру и длину.

**Идея метода**
Авторы предложили объединить масштабное обучение на "слабо размеченных" данных (Weakly-supervised) с механизмом **Instructions**, который явно указывает модели роль входного текста. Вместо того чтобы создавать разные башни для запросов и документов, используется один энкодер, но к тексту добавляется префикс `query: ` или `passage: `.

**Существующие альтернативы**
* **DPR (2020)**: использовал две разные BERT-башни. Проблема: требует сложных техник Hard Negative Mining и плохо работает Zero-shot на новых доменах.
* **SimCSE (2021)**: обучался через Contrastive Learning, используя Dropout как аугментацию. Проблема: оптимизирован для симметричных задач (предложение-предложение), не учитывает специфику "короткий запрос — длинный документ".
* **Contriever (2021)**: использовал Contrastive Pre-training на больших корпусах. Проблема: отсутствие явного разделения ролей запроса и документа приводило к субоптимальным результатам в Retrieval.

**Архитектура**
E5 базируется на стандартной архитектуре Transformer Encoder (на основе модели **RoBERTa**, 2019). 
1. **Input Representation**: К каждому входу добавляется префикс. Для запросов — `query: {text}`, для документов — `passage: {text}`.
2. **Pooling**: Используется Mean Pooling по всем токенам выходного слоя для получения финального вектора.
3. **Similarity**: Косинусное расстояние между эмбеддингами.

**Алгоритм обучения**
Процесс разбит на две стадии:

1. **Weakly-supervised Pre-training**:
   * Используется датасет **CCPairs** (270 млн пар), собранный из веба (заголовки постов — основной текст, вопросы — ответы и т.д.).
   * Применяется Contrastive Loss (InfoNCE).
   * Новизна: вместо классического Hard Negative Mining используется очень большой Batch Size (32k), что позволяет модели видеть достаточное количество "естественных" негативных примеров в рамках одного батча.

2. **Supervised Fine-tuning**:
   * Модель дообучается на смеси высококачественных наборов данных (MS MARCO, NLI, Natural Questions).
   * Здесь уже применяется Hard Negative Mining (выбор наиболее похожих, но нерелевантных документов) для уточнения границ признакового пространства.

**Алгоритм инференса**
1. Если нужно проиндексировать базу знаний: к каждому документу добавляется префикс `passage: `, прогоняется через Encoder, эмбеддинг сохраняется в ANN-индекс (например, FAISS).
2. При поиске: к запросу добавляется префикс `query: `, генерируется эмбеддинг.
3. Выполняется поиск по косинусному расстоянию в индексе.

**Результаты**
* **MTEB Benchmark**: На момент выхода E5 заняла первое место в бенчмарке MTEB (Massive Text Embedding Benchmark), обойдя проприетарные модели от OpenAI.
* **Zero-shot Retrieval**: На наборе BEIR модель E5-large показала точность (NDCG@10) 54.3%, что на 5-7 п.п. выше, чем у моделей аналогичного размера, обученных без использования префиксов и этапа слабого обучения на CCPairs.
* **Эффективность**: Благодаря тому, что модель использует одну башню (Single-tower) и стандартный BERT-layout, она полностью совместима с существующей инфраструктурой для трансформеров и не требует специфических слоев.

## 📝 Критический анализ

```markdown
# E5 (2022)
---
[[paper]](https://arxiv.org/abs/2212.03533)<br>
**E5** = **E**mb**E**ddings from bidirection **E**ncoder representations

E5 — семейство моделей (Small, Base, Large) для генерации универсальных Text Embeddings от Microsoft. Метод впервые показал, как использование текстовых инструкций-префиксов в Encoder решает проблему асимметричного поиска.

**Контекст**
До E5 модели для Dense Retrieval обучались на узких supervised-датасетах (DPR) или через Unsupervised Contrastive Learning (SimCSE), что плохо работало для поиска, где запрос и документ различны по структуре и длине.

<img src="img/img.png" width=500>

**Идея метода**
Объединение масштабного обучения на "слабо размеченных" данных с механизмом **Instructions**. Вместо разных башен для запросов и документов используется один энкодер с префиксами `query: ` или `passage: `.

**Существующие альтернативы**
* **DPR (2020)**: две BERT-башни, сложные техники Hard Negative Mining, плохая Zero-shot производительность.
* **SimCSE (2021)**: Contrastive Learning с Dropout, оптимизирован для симметричных задач.
* **Contriever (2021)**: Contrastive Pre-training, отсутствие явного разделения ролей запроса и документа.

**Архитектура**
E5 базируется на Transformer Encoder (RoBERTa, 2019).
1. **Input Representation**: Префиксы для запросов и документов.
2. **Pooling**: Mean Pooling для финального вектора.
3. **Similarity**: Косинусное расстояние между эмбеддингами.

**Алгоритм обучения**
1. **Weakly-supervised Pre-training**:
   * Датасет **CCPairs** (270 млн пар).
   * Contrastive Loss (InfoNCE) с большим Batch Size (32k).

2. **Supervised Fine-tuning**:
   * Дообучение на MS MARCO, NLI, Natural Questions.
   * Hard Negative Mining для уточнения признакового пространства.

**Алгоритм инференса**
1. Индексация: префикс `passage: `, Encoder, эмбеддинг в ANN-индекс (например, FAISS).
2. Поиск: префикс `query: `, генерация эмбеддинга.
3. Поиск по косинусному расстоянию.

**Результаты**
* **MTEB Benchmark**: E5 заняла первое место, обойдя модели OpenAI.
* **Zero-shot Retrieval**: На BEIR E5-large показала NDCG@10 54.3%, на 5-7 п.п. выше аналогов.
* **Эффективность**: Использует одну башню и стандартный BERT-layout, совместима с существующей инфраструктурой.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Загрузим предобученную модель и токенизатор E5
# В реальном сценарии, вы бы использовали модель, обученную на основе E5, но для иллюстрации
# мы используем стандартную модель RoBERTa, так как E5 основана на этой архитектуре.
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Функция для генерации эмбеддингов с использованием префиксов
def generate_embedding(text, prefix):
    # Добавляем префикс к тексту
    prefixed_text = f"{prefix}: {text}"
    # Токенизируем текст
    inputs = tokenizer(prefixed_text, return_tensors="pt", truncation=True, padding=True)
    # Пропускаем через модель
    outputs = model(**inputs)
    # Используем Mean Pooling для получения финального эмбеддинга
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Пример текста запроса и документа
query_text = "What is the capital of France?"
document_text = "Paris is the capital city of France, known for its art, fashion, and culture."

# Генерируем эмбеддинги для запроса и документа
query_embedding = generate_embedding(query_text, "query")
document_embedding = generate_embedding(document_text, "passage")

# Вычисляем косинусное сходство между эмбеддингами
similarity_score = cosine_similarity(query_embedding.detach().numpy(), document_embedding.detach().numpy())

# Выводим результат
print(f"Cosine Similarity between query and document: {similarity_score[0][0]}")

# Пример использования большого батча для слабого обучения
# В реальном сценарии, вы бы использовали датасет CCPairs, но здесь мы создадим игрушечный пример
batch_texts = [
    "query: What is the capital of Germany?",
    "passage: Berlin is the capital of Germany.",
    "query: Who wrote '1984'?",
    "passage: George Orwell wrote the novel '1984'.",
    # Добавляем больше примеров для иллюстрации большого батча
]

# Токенизируем и пропускаем через модель
inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
# Используем Mean Pooling
embeddings = outputs.last_hidden_state.mean(dim=1)

# Вычисляем косинусное сходство между всеми парами в батче
similarity_matrix = cosine_similarity(embeddings.detach().numpy())

# Выводим матрицу сходства
print("Cosine Similarity Matrix:")
print(similarity_matrix)

# В реальном сценарии, для слабого обучения использовался бы Contrastive Loss (например, InfoNCE)
# с большим батчем, чтобы модель видела много "естественных" негативных примеров.
```

### Комментарии к коду:
1. **Префиксы**: Мы добавляем префиксы `query:` и `passage:` к текстам, чтобы явно указать модели роль входного текста. Это ключевая особенность метода E5.
2. **Mean Pooling**: Используем среднее значение по всем токенам выходного слоя для получения финального эмбеддинга, как описано в архитектуре E5.
3. **Косинусное сходство**: Вычисляем косинусное сходство между эмбеддингами запроса и документа для оценки их семантической близости.
4. **Большой батч**: Иллюстрируем идею использования большого батча для слабого обучения, что позволяет модели видеть множество "естественных" негативных примеров без сложных техник Hard Negative Mining.